# 01_roles — Authorization with role classes

Roles in AOA are **classes, not strings**. Business roles subclass `ApplicationRole`; privilege inheritance follows Python subclassing — access is granted via `issubclass(user_role, required_role)`.

This notebook declares a role hierarchy (`AdminRole` is-a `ManagerRole`) and three actions guarded differently, then runs them under three principals (anonymous, manager, admin) and prints the allow/deny matrix.

**What's new**

| Concept | Description |
|---------|-------------|
| `@check_roles(...)` | Required access declaration on the Action (one of three mandatory decorators) |
| `ApplicationRole` | Base class for business roles — `name`, `description`, subclassing |
| `NoneRole` | Engine sentinel: open to everyone (stated explicitly; silence is not allowed) |
| role hierarchy | `AdminRole(ManagerRole)` → an admin counts as a manager |
| `Context(user=UserInfo(roles=(...)))` | Where the caller's roles live |
| `AuthorizationError` | Raised on denial — before any aspect runs |

In [ ]:
!pip install aoa-action-machine

In [ ]:
import asyncio

from pydantic import Field

from aoa.action_machine.auth import ApplicationRole, NoneRole
from aoa.action_machine.context import Context
from aoa.action_machine.context.user_info import UserInfo
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.exceptions.authorization_error import AuthorizationError
from aoa.action_machine.intents.aspects import summary_aspect
from aoa.action_machine.intents.check_roles import check_roles
from aoa.action_machine.intents.meta import meta
from aoa.action_machine.model import BaseAction, BaseParams, BaseResult
from aoa.action_machine.runtime.action_product_machine import ActionProductMachine

## Domain

In [ ]:
class StoreDomain(BaseDomain):
    name = "store"
    description = "Store domain"

## Role hierarchy

`AdminRole` subclasses `ManagerRole`, so an admin counts as a manager: `issubclass(AdminRole, ManagerRole)` is `True`. Business roles inherit `RoleMode.ALIVE` from `ApplicationRole` — no `@role_mode` needed for live roles. The class name must end with `Role`.

In [ ]:
class ManagerRole(ApplicationRole):
    name = "manager"
    description = "Can manage orders"


class AdminRole(ManagerRole):
    name = "admin"
    description = "Full control; includes manager privileges"

## Params & Result

In [ ]:
class OrderParams(BaseParams):
    order_id: str = Field(description="Order identifier")


class OrderResult(BaseResult):
    order_id: str = Field(description="Order identifier")
    action: str = Field(description="What was performed")

## Three actions, three access policies

- `NoneRole` — open to everyone (stated explicitly; silence is not allowed)
- `ManagerRole` — managers and, by inheritance, admins
- `AdminRole` — admins only

In [ ]:
@meta(description="View an order", domain=StoreDomain)
@check_roles(NoneRole)
class GetOrderAction(BaseAction[OrderParams, OrderResult]):

    @summary_aspect("Return the order")
    async def get_summary(self, params, state, box, connections):
        return OrderResult(order_id=params.order_id, action="viewed")


@meta(description="Cancel an order", domain=StoreDomain)
@check_roles(ManagerRole)
class CancelOrderAction(BaseAction[OrderParams, OrderResult]):

    @summary_aspect("Cancel the order")
    async def cancel_summary(self, params, state, box, connections):
        return OrderResult(order_id=params.order_id, action="cancelled")


@meta(description="Purge all orders", domain=StoreDomain)
@check_roles(AdminRole)
class PurgeOrdersAction(BaseAction[OrderParams, OrderResult]):

    @summary_aspect("Purge orders")
    async def purge_summary(self, params, state, box, connections):
        return OrderResult(order_id=params.order_id, action="purged")

## Run — the access matrix

The same three actions run under three principals. On denial the machine raises `AuthorizationError` **before** any aspect runs.

> In Colab, `await` works at top level — no `asyncio.run()`.

In [ ]:
async def main() -> None:
    machine = ActionProductMachine()

    principals = [
        ("anonymous", Context()),
        ("manager", Context(user=UserInfo(user_id="m1", roles=(ManagerRole,)))),
        ("admin", Context(user=UserInfo(user_id="a1", roles=(AdminRole,)))),
    ]
    actions = [
        ("GetOrder    [NoneRole]", GetOrderAction),
        ("CancelOrder [ManagerRole]", CancelOrderAction),
        ("PurgeOrders [AdminRole]", PurgeOrdersAction),
    ]

    for user_name, ctx in principals:
        print(f"\nUser: {user_name}")
        for action_name, action_cls in actions:
            try:
                await machine.run(ctx, action_cls(), OrderParams(order_id="ord-001"))
                print(f"  {action_name:<26} -> allowed")
            except AuthorizationError:
                print(f"  {action_name:<26} -> denied")


await main()